In [1]:
import pandas as pd

df = pd.read_csv("churn-bigml-80.csv")

df.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [2]:
df.shape

(2666, 20)

In [3]:
df.columns

Index(['State', 'Account length', 'Area code', 'International plan',
       'Voice mail plan', 'Number vmail messages', 'Total day minutes',
       'Total day calls', 'Total day charge', 'Total eve minutes',
       'Total eve calls', 'Total eve charge', 'Total night minutes',
       'Total night calls', 'Total night charge', 'Total intl minutes',
       'Total intl calls', 'Total intl charge', 'Customer service calls',
       'Churn'],
      dtype='object')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2666 entries, 0 to 2665
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   2666 non-null   object 
 1   Account length          2666 non-null   int64  
 2   Area code               2666 non-null   int64  
 3   International plan      2666 non-null   object 
 4   Voice mail plan         2666 non-null   object 
 5   Number vmail messages   2666 non-null   int64  
 6   Total day minutes       2666 non-null   float64
 7   Total day calls         2666 non-null   int64  
 8   Total day charge        2666 non-null   float64
 9   Total eve minutes       2666 non-null   float64
 10  Total eve calls         2666 non-null   int64  
 11  Total eve charge        2666 non-null   float64
 12  Total night minutes     2666 non-null   float64
 13  Total night calls       2666 non-null   int64  
 14  Total night charge      2666 non-null   

In [5]:
df_clean = df.copy()

In [6]:
df_clean.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


In [7]:
df_clean["Churn"].value_counts(normalize=True) * 100

Churn
False    85.446362
True     14.553638
Name: proportion, dtype: float64

In [8]:
df_clean.groupby("International plan")["Churn"].mean()

International plan
No     0.112688
Yes    0.437037
Name: Churn, dtype: float64

In [9]:
df_clean.groupby("Customer service calls")["Churn"].mean()

Customer service calls
0    0.142342
1    0.104762
2    0.101974
3    0.106322
4    0.481203
5    0.591837
6    0.588235
7    0.625000
8    1.000000
9    1.000000
Name: Churn, dtype: float64

In [11]:
df_clean.columns

Index(['State', 'Account length', 'Area code', 'International plan',
       'Voice mail plan', 'Number vmail messages', 'Total day minutes',
       'Total day calls', 'Total day charge', 'Total eve minutes',
       'Total eve calls', 'Total eve charge', 'Total night minutes',
       'Total night calls', 'Total night charge', 'Total intl minutes',
       'Total intl calls', 'Total intl charge', 'Customer service calls',
       'Churn'],
      dtype='object')

In [12]:
df_clean["Total_minutes"] = (
    df_clean["Total day minutes"] +
    df_clean["Total eve minutes"] +
    df_clean["Total night minutes"] +
    df_clean["Total intl minutes"]
)

In [13]:
df_clean.columns

Index(['State', 'Account length', 'Area code', 'International plan',
       'Voice mail plan', 'Number vmail messages', 'Total day minutes',
       'Total day calls', 'Total day charge', 'Total eve minutes',
       'Total eve calls', 'Total eve charge', 'Total night minutes',
       'Total night calls', 'Total night charge', 'Total intl minutes',
       'Total intl calls', 'Total intl charge', 'Customer service calls',
       'Churn', 'Total_minutes'],
      dtype='object')

In [14]:
df_clean.groupby("Churn")["Total_minutes"].mean()

Churn
False    584.559658
True     630.693041
Name: Total_minutes, dtype: float64

In [15]:
df_clean.groupby("Churn")["Customer service calls"].mean()

Churn
False    1.453029
True     2.206186
Name: Customer service calls, dtype: float64

In [16]:
def churn_risk(row):
    score = 0
    
    # International plan risk
    if row["International plan"] == "Yes":
        score += 2
    
    # Customer service calls risk
    if row["Customer service calls"] >= 4:
        score += 3
    elif row["Customer service calls"] >= 2:
        score += 1
    
    # High usage risk (relative indicator)
    if row["Total_minutes"] > df_clean["Total_minutes"].mean():
        score += 1
    
    return score

df_clean["Risk_Score"] = df_clean.apply(churn_risk, axis=1)

In [17]:
def risk_level(score):
    if score >= 4:
        return "High Risk"
    elif score >= 2:
        return "Medium Risk"
    else:
        return "Low Risk"

df_clean["Risk_Level"] = df_clean["Risk_Score"].apply(risk_level)

In [18]:
df_clean["Risk_Level"].value_counts()

Risk_Level
Low Risk       1772
Medium Risk     742
High Risk       152
Name: count, dtype: int64

In [19]:
df_clean.groupby("Risk_Level")["Churn"].mean()

Risk_Level
High Risk      0.328947
Low Risk       0.066027
Medium Risk    0.297844
Name: Churn, dtype: float64

In [20]:
df_clean.to_csv("churn_cleaned.csv", index=False)

In [21]:
import os
os.listdir()

['.ipynb_checkpoints',
 'churn-bigml-20.csv',
 'churn-bigml-80.csv',
 'churn_cleaned.csv',
 'Customer_Churn_Analysis.ipynb']

In [22]:
import sqlite3
conn = sqlite3.connect("churn.db")
df_clean.to_sql("churn_data", conn, if_exists="replace", index=False)

2666

In [23]:
pd.read_sql("SELECT * FROM churn_data LIMIT 5", conn)

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,...,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn,Total_minutes,Risk_Score,Risk_Level
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,...,91,11.01,10.0,3,2.70,1,0,717.2,1,Low Risk
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,...,103,11.45,13.7,3,3.70,1,0,625.2,1,Low Risk
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,...,104,7.32,12.2,5,3.29,0,0,539.4,0,Low Risk
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,...,89,8.86,6.6,7,1.78,2,0,564.8,3,Medium Risk
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,...,121,8.41,10.1,3,2.73,3,0,512.0,3,Medium Risk


In [24]:
pd.read_sql("""
SELECT 
    AVG(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) * 100 AS churn_rate
FROM churn_data
""", conn)

,churn_rate
0,14.553638


In [25]:
pd.read_sql("""
SELECT 
    CASE 
        WHEN "Account length" <= 50 THEN '0-50'
        WHEN "Account length" <= 100 THEN '51-100'
        WHEN "Account length" <= 150 THEN '101-150'
        ELSE '150+'
    END AS tenure_group,
    AVG(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) * 100 AS churn_rate
FROM churn_data
GROUP BY tenure_group
ORDER BY tenure_group
""", conn)

,tenure_group,churn_rate
0,0-50,13.309353
1,101-150,14.747859
2,150+,16.423358
3,51-100,14.205080


In [26]:
pd.read_sql("""
SELECT 
    "International plan",
    AVG(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) * 100 AS churn_rate
FROM churn_data
GROUP BY "International plan"
""", conn)

,International plan,churn_rate
0,No,11.268781
1,Yes,43.703704


In [27]:
pd.read_sql("""
SELECT 
    "Voice mail plan",
    AVG(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) * 100 AS churn_rate
FROM churn_data
GROUP BY "Voice mail plan"
""", conn)

,Voice mail plan,churn_rate
0,No,16.709778
1,Yes,8.867667


In [28]:
pd.read_sql("""
SELECT 
    Churn,
    AVG("Total day minutes") AS avg_day_minutes,
    AVG("Total eve minutes") AS avg_eve_minutes,
    AVG("Total night minutes") AS avg_night_minutes
FROM churn_data
GROUP BY Churn
""", conn)

,Churn,avg_day_minutes,avg_eve_minutes,avg_night_minutes
0,0,175.104346,198.853380,200.464091
1,1,205.181186,209.385309,205.307216


In [29]:
pd.read_sql("""
SELECT 
    Churn,
    AVG("Total day charge" + "Total eve charge" + "Total night charge" + "Total intl charge") AS avg_revenue
FROM churn_data
GROUP BY Churn
""", conn)

,Churn,avg_revenue
0,0,58.429759
1,1,64.839820


In [30]:
pd.read_sql("""
SELECT 
    CASE 
        WHEN "Customer service calls" >= 4 
             OR "International plan" = 'Yes'
             OR ("Total day minutes" + "Total eve minutes" + "Total night minutes" + "Total intl minutes") > 
                (SELECT AVG("Total day minutes" + "Total eve minutes" + "Total night minutes" + "Total intl minutes") FROM churn_data)
        THEN 'High Risk'
        
        WHEN "Customer service calls" >= 2 
             OR "International plan" = 'No'
        THEN 'Medium Risk'
        
        ELSE 'Low Risk'
    END AS churn_segment,
    
    COUNT(*) AS total_customers,
    AVG(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) * 100 AS churn_rate,
    AVG("Total day charge" + "Total eve charge" + "Total night charge" + "Total intl charge") AS avg_revenue
    
FROM churn_data
GROUP BY churn_segment
""", conn)

,churn_segment,total_customers,churn_rate,avg_revenue
0,High Risk,1583,22.994315,64.570379
1,Medium Risk,1083,2.216066,51.750628


In [31]:
pd.read_sql("""
SELECT 
    CASE 
        WHEN "Customer service calls" >= 4 
             OR "International plan" = 'Yes'
             OR ("Total day minutes" + "Total eve minutes" + "Total night minutes" + "Total intl minutes") > 
                (SELECT AVG("Total day minutes" + "Total eve minutes" + "Total night minutes" + "Total intl minutes") FROM churn_data)
        THEN 'High Risk'

        WHEN "Customer service calls" >= 2 
             OR "International plan" = 'No'
        THEN 'Medium Risk'

        ELSE 'Low Risk'
    END AS churn_segment,

    COUNT(*) AS total_customers,
    AVG(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) * 100 AS churn_rate,
    AVG("Total day charge" + "Total eve charge" + "Total night charge" + "Total intl charge") AS avg_revenue

FROM churn_data
GROUP BY churn_segment
""", conn)

,churn_segment,total_customers,churn_rate,avg_revenue
0,High Risk,1583,22.994315,64.570379
1,Medium Risk,1083,2.216066,51.750628


In [32]:
pd.read_sql("""
SELECT 
    CASE 
        WHEN "Customer service calls" >= 4 
             OR "International plan" = 'Yes'
             OR ("Total day minutes" + "Total eve minutes" + "Total night minutes" + "Total intl minutes") >
                (SELECT AVG("Total day minutes" + "Total eve minutes" + "Total night minutes" + "Total intl minutes") 
                 FROM churn_data)
        THEN 'High Risk'

        WHEN "Customer service calls" >= 2 
        OR ("Total day minutes" + "Total eve minutes" + "Total night minutes" + "Total intl minutes") >
           (SELECT AVG("Total day minutes" + "Total eve minutes" + "Total night minutes" + "Total intl minutes") 
            FROM churn_data)
        THEN 'Medium Risk'

        ELSE 'Low Risk'
    END AS churn_segment,

    COUNT(*) AS total_customers,
    AVG(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) * 100 AS churn_rate,
    AVG("Total day charge" + "Total eve charge" + "Total night charge" + "Total intl charge") AS avg_revenue

FROM churn_data
GROUP BY churn_segment
""", conn)

,churn_segment,total_customers,churn_rate,avg_revenue
0,High Risk,1583,22.994315,64.570379
1,Low Risk,646,2.476780,51.823560
2,Medium Risk,437,1.830664,51.642815


In [33]:
df_clean.to_csv("churn_powerbi_ready.csv", index=False)

In [34]:
import os
os.listdir()

['.ipynb_checkpoints',
 'churn-bigml-20.csv',
 'churn-bigml-80.csv',
 'churn.db',
 'churn_cleaned.csv',
 'churn_powerbi_ready.csv',
 'Customer_Churn_Analysis.ipynb']